In [3]:
"""
4-Way Ablation Study: base / RAG-only / LoRA-only / combined
Microfluidics + Bilirubin Detection RAFT Research Assistant

Run this in a Kaggle T4 (or T4x2) notebook. Structured as sequential cells
(marked with # %% CELL) so you can paste into separate notebook cells or
run as a script end-to-end.

Design notes (read before running):
- Only ONE 7B fp16 model is held in memory at a time. Base model runs
  conditions {base, rag_only}, then gets released, then the merged
  fine-tune runs {lora_only, combined}. This is why results are grouped
  by model-load, not by question.
- LoRA-only = fine-tuned model, NO retrieval context, NO RAFT doc-block
  wrapper -- i.e. "what did the weights learn on their own". If you'd
  rather test the fine-tune with an empty-context RAFT-shaped prompt
  instead, flip LORA_ONLY_USE_EMPTY_TEMPLATE to True below.
- Retrieval-score threshold (0.4) and post-hoc repeat-sentence truncation
  are carried over from your validated RAG pipeline, applied identically
  to rag_only and combined so the only variable between them is the base
  vs fine-tuned model.
"""

import os
 
# Must be set before torch initializes CUDA -- reduces OOM from memory
# fragmentation on a tight-VRAM T4 running an unquantized 7B model.
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
 
import gc
import json
import re
import time
from pathlib import Path
 
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

In [1]:
!pip install faiss-cpu gradio pdfplumber -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 2.8 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.1/69.1 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 74.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.0/60.0 kB 4.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 105.8 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.7/3.7 MB 95.3 MB/s eta 0:00:00:00:01


In [4]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
from huggingface_hub import login
login() 

In [ ]:
# %% CELL 1 -- CONFIG ---------------------------------------------------

BASE_MODEL_ID = "mistralai/Mistral-7B-Instruct-v0.2"         
FT_MODEL_ID   = "aryxnsinhx/mistral-7b-microfluidics-raft"  

RETRIEVAL_SCORE_THRESHOLD = 0.4
TOP_K = 5  
MAX_CHUNK_CHARS_IN_PROMPT = 700 
LORA_ONLY_USE_EMPTY_TEMPLATE = False  

GEN_KWARGS = dict(
    do_sample=False,
    repetition_penalty=1.1,
    max_new_tokens=350,
)

OUT_DIR = Path("/kaggle/working/ablation_results")
OUT_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
QUESTIONS = [
    # --- Core domain fluency (in-corpus, single concept) ---
    {
        "id": "F1", "category": "fluency",
        "question": "What physical mechanism drives mixing in a microfluidic channel at low Reynolds number, and why can't turbulence be relied on?",
        "expect": "grounded_answer",
    },
    {
        "id": "F2", "category": "fluency",
        "question": "What causes electrohydrodynamic instability at a liquid-liquid interface under an applied electric field?",
        "expect": "grounded_answer",
    },
    {
        "id": "F3", "category": "fluency",
        "question": "Walk through the typical PDMS soft lithography fabrication steps used to make a microfluidic device.",
        "expect": "grounded_answer",
    },
    {
        "id": "F4", "category": "fluency",
        "question": "How is a diazonium salt used to detect bilirubin in a sample, chemically speaking?",
        "expect": "grounded_answer",
    },
    {
        "id": "F5", "category": "fluency",
        "question": "What role does channel geometry (e.g. herringbone or serpentine patterns) play in passive micromixer design?",
        "expect": "grounded_answer",
    },

    # --- Eponym-mismatch nuance ---
    {
        "id": "E1", "category": "eponym",
        "question": "Explain the Van den Bergh reaction and how it's used to distinguish conjugated from unconjugated bilirubin.",
        "expect": "abstain",  # name not present in corpus even though the chemistry is
        "note": "Same chemistry as F4 but asked via the eponym absent from source text.",
    },
    {
        "id": "E2", "category": "eponym",
        "question": "What does the diazo reaction tell you about the conjugation state of bilirubin in a blood sample?",
        "expect": "grounded_answer",
        "note": "Paraphrase of E1 using terminology that IS present in the corpus -- should NOT abstain.",
    },
    {
        "id": "E3", "category": "eponym",
        "question": "How does the Taylor-Aris dispersion phenomenon apply to analyte transport in your microfluidic channels?",
        "expect": "abstain_or_grounded",
        "note": "Check manually whether this specific named phenomenon appears in corpus text; adjust expect field once verified.",
    },

    # --- Clean abstention probes (genuinely out-of-scope) ---
    {
        "id": "A1", "category": "abstention",
        "question": "What is the optimal CRISPR guide RNA design for editing the UGT1A1 gene to treat Gilbert's syndrome?",
        "expect": "abstain",
    },
    {
        "id": "A2", "category": "abstention",
        "question": "Describe the fictional 'quantum microfluidic teleportation channel' method for instant sample transport.",
        "expect": "abstain",
    },
    {
        "id": "A3", "category": "abstention",
        "question": "What was the GDP impact of microfluidic diagnostic device adoption in the US healthcare market in 2023?",
        "expect": "abstain",
    },

    # --- Multi-paper synthesis (needs 2+ source papers) ---
    {
        "id": "S1", "category": "synthesis",
        "question": "How does the mixing efficiency described in 'Current methods for characterising mixing and flow in microchannels' relate to the detection sensitivity implied by 'KINETICS OF THE FORMATION OF AZOBILIRUBIN, DETERMINATION OF THE RATE CONSTANT K OF THE SECOND REACTION'?",
        "expect": "grounded_answer",
        "note": "Real titles confirmed from retrieval hits during earlier run (top scores 0.549/0.431 respectively for the placeholder version) -- verify exact title strings match your metadata before trusting fully.",
    },
    {
        "id": "S2", "category": "synthesis",
        "question": "Compare the PDMS fabrication considerations in 'Microfluidic devices fabricated in poly(dimethylsiloxane) for biological studies' against the channel requirements implied by 'Electric field mediated von Karman vortices in stratified microflows transition from linear instabilities to coherent mixing'. Are they compatible?",
        "expect": "grounded_answer",
        "note": "Real titles confirmed from retrieval hits during earlier run. Double-check the von Karman paper's exact title string (diacritics/wording) matches your metadata exactly -- title strings must match for retrieval to work correctly if you later use them in filtering.",
    },
    {
        "id": "S3", "category": "synthesis",
        "question": "Synthesize across the corpus: what design trade-offs connect channel geometry, mixing time, and colorimetric bilirubin detection accuracy?",
        "expect": "grounded_answer",
        "note": "Broad synthesis question, deliberately not paper-scoped -- tests whether retrieval pulls from multiple sources. Prior run: combined model declined here even with retrieval above threshold (0.434) -- top-3 chunks were topically adjacent (mixing methods, azobilirubin kinetics) but didn't state an explicit connection. Worth treating this abstention as plausibly CORRECT calibration rather than a failure -- re-evaluate after TOP_K bump.",
    },

    # --- Isolation set: separating "named-source" trigger from genuine synthesis difficulty ---
    # S1/S2 (above) both explicitly name two source papers in the question and
    # both showed combined falsely claiming the docs weren't present, even
    # though retrieval_hits confirmed both papers WERE in context (rag_only
    # on the identical context answered correctly). These three questions
    # hold the underlying synthesis intent constant while varying how many
    # papers are explicitly named in the question text, to isolate whether
    # "named-source" phrasing itself is the trigger.
    {
        "id": "S4", "category": "synthesis",
        "question": "How might mixing efficiency in a microchannel affect the sensitivity of a colorimetric bilirubin detection assay?",
        "expect": "grounded_answer",
        "note": "Same synthesis intent as S1, ZERO papers named explicitly. If combined answers this correctly, the trigger is specifically named-source phrasing, not synthesis difficulty itself.",
    },
    {
        "id": "S5", "category": "synthesis",
        "question": "According to 'Current methods for characterising mixing and flow in microchannels', how is mixing efficiency typically assessed?",
        "expect": "grounded_answer",
        "note": "Single named paper, single-source question (not a cross-paper synthesis question at all). If combined STILL falsely abstains here, the trigger is 'any named source' rather than specifically 'two named sources being compared'.",
    },
    {
        "id": "S6", "category": "synthesis",
        "question": "Do the PDMS fabrication considerations discussed in the corpus intersect with the channel dimension requirements needed for electrohydrodynamic instability effects to occur?",
        "expect": "grounded_answer",
        "note": "Same synthesis intent as S2, ZERO papers named explicitly. Direct paired comparison against S2 to isolate the naming-trigger effect.",
    },
]

assert len(QUESTIONS) >= 10, "expand question bank to at least 10-15 as planned"

In [ ]:

#RETRIEVAL (reuse your existing FAISS index) ---------------

DATASET_DIR = "/kaggle/input/datasets/aryxnsinhx1/ablation-aryxn"
INDEX_PATH = f"{DATASET_DIR}/microfluidics.index"
METADATA_PATH = f"{DATASET_DIR}/chunk_metadata.pkl"


def load_retriever():
    """
    Load your existing FAISS index + all-mpnet-base-v2 embedder.
    Assumes the schema you validated: chunk_id, paper_title, section,
    text, source_file. Metadata comes from chunk_metadata.pkl (pickled
    pandas DataFrame or list of dicts) -- chunks.jsonl is raw first-stage
    chunking output and is not used here.
    """
    import pickle

    import faiss
    from sentence_transformers import SentenceTransformer

    embedder = SentenceTransformer("all-mpnet-base-v2", device="cpu")
    index = faiss.read_index(INDEX_PATH)

    with open(METADATA_PATH, "rb") as f:
        raw = pickle.load(f)

    chunks = raw.to_dict(orient="records") if hasattr(raw, "to_dict") else list(raw)

    required = {"chunk_id", "paper_title", "section", "text", "source_file"}
    missing = required - set(chunks[0].keys())
    if missing:
        print(f"WARNING: metadata missing expected fields: {missing}. "
              f"Actual fields present: {list(chunks[0].keys())}")

    print(f"Loaded {index.ntotal} vectors from index, {len(chunks)} chunk records from metadata.")
    if index.ntotal != len(chunks):
        print(f"WARNING: index size ({index.ntotal}) != chunk count ({len(chunks)}) "
              f"-- ordering between FAISS index and metadata list must match by "
              f"position, double check this before trusting retrieval results.")

    return embedder, index, chunks


def retrieve(query, embedder, index, chunks, top_k=TOP_K):
    q_emb = embedder.encode([query], normalize_embeddings=True)
    scores, idxs = index.search(np.array(q_emb, dtype="float32"), top_k)
    hits = []
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        c = chunks[idx]
        hits.append({
            "score": float(score),
            "paper_title": c["paper_title"],
            "section": c["section"],
            "text": c["text"],
        })
    return hits


def format_doc_block(hits):
    """
    Matches training format exactly: [DOC N] (paper_title, section) text
    Truncates chunk text to MAX_CHUNK_CHARS_IN_PROMPT to keep the prompt
    short enough to fit generation activations on a T4 -- this only
    affects what the model sees, not the full chunk text stored in
    `hits`/retrieval_hits for scoring/citation purposes.
    """
    blocks = []
    for i, h in enumerate(hits, start=1):
        text = h["text"]
        if len(text) > MAX_CHUNK_CHARS_IN_PROMPT:
            text = text[:MAX_CHUNK_CHARS_IN_PROMPT].rsplit(" ", 1)[0] + "..."
        blocks.append(f"[DOC {i}] ({h['paper_title']}, {h['section']}) {text}")
    return "\n\n".join(blocks)

In [ ]:
#GENERATION HELPERS ----------------------------------------

def load_model(model_id):
    tok = AutoTokenizer.from_pretrained(model_id)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=torch.float16,
        device_map="auto",
        low_cpu_mem_usage=True,
        attn_implementation="sdpa",  # more memory-efficient attention than eager
    )
    model.eval()
    return tok, model


def release_model(tok, model):
    del model
    del tok
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()


def truncate_at_first_repeat(text):
    """Post-hoc safety net: cut output at the first sentence that repeats."""
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    seen = set()
    kept = []
    for s in sentences:
        key = s.strip().lower()
        if key in seen and len(key) > 15:
            break
        seen.add(key)
        kept.append(s)
    return " ".join(kept)


def build_prompt(question, doc_block=None):
    if doc_block:
        return (
            "You are a research assistant. Use only the context below to answer. "
            "If the context does not contain the answer, say you don't know.\n\n"
            f"Context:\n{doc_block}\n\nQuestion: {question}\n[/INST]"
        )
    return f"Question: {question}\n[/INST]"


def generate(tok, model, prompt):
    """
    Returns (answer_text, error). error is None on success, or a short
    string describing an OOM/other failure so the caller can log it and
    keep going instead of crashing the whole run.
    """
    try:
        inputs = tok(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, **GEN_KWARGS, pad_token_id=tok.eos_token_id)
        text = tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return truncate_at_first_repeat(text), None
    except torch.cuda.OutOfMemoryError as e:
        return None, f"OOM: {e}"
    finally:
        try:
            del inputs
        except NameError:
            pass
        try:
            del out
        except NameError:
            pass
        gc.collect()
        torch.cuda.empty_cache()


In [ ]:
#RUN: BASE MODEL (conditions: base, rag_only) --------------

def run_base_conditions(embedder, index, chunks):
    tok, model = load_model(BASE_MODEL_ID)
    results = []

    for q in QUESTIONS:
        # condition: base (no context at all)
        prompt = build_prompt(q["question"])
        t0 = time.time()
        answer, err = generate(tok, model, prompt)
        results.append({
            "id": q["id"], "category": q["category"], "condition": "base",
            "question": q["question"],
            "answer": answer if err is None else f"[GENERATION ERROR: {err}]",
            "retrieval_hits": None, "elapsed_s": round(time.time() - t0, 1),
        })

        # condition: rag_only (base model + retrieved context)
        hits = retrieve(q["question"], embedder, index, chunks)
        top_score = hits[0]["score"] if hits else 0.0
        if top_score < RETRIEVAL_SCORE_THRESHOLD:
            answer, err = "[ABSTAINED: retrieval score below threshold]", None
        else:
            doc_block = format_doc_block(hits)
            prompt = build_prompt(q["question"], doc_block)
            t0 = time.time()
            answer, err = generate(tok, model, prompt)
        results.append({
            "id": q["id"], "category": q["category"], "condition": "rag_only",
            "question": q["question"],
            "answer": answer if err is None else f"[GENERATION ERROR: {err}]",
            "retrieval_hits": hits, "top_score": round(top_score, 3),
            "elapsed_s": round(time.time() - t0, 1),
        })

    release_model(tok, model)
    return results

In [ ]:
#RUN: FINE-TUNED MODEL (conditions: lora_only, combined) ---

def run_finetuned_conditions(embedder, index, chunks):
    tok, model = load_model(FT_MODEL_ID)
    results = []

    for q in QUESTIONS:
        # condition: lora_only (fine-tuned weights, no retrieval)
        if LORA_ONLY_USE_EMPTY_TEMPLATE:
            prompt = build_prompt(q["question"], doc_block="[DOC 1] (none, none) ")
        else:
            prompt = build_prompt(q["question"])
        t0 = time.time()
        answer, err = generate(tok, model, prompt)
        results.append({
            "id": q["id"], "category": q["category"], "condition": "lora_only",
            "question": q["question"],
            "answer": answer if err is None else f"[GENERATION ERROR: {err}]",
            "retrieval_hits": None, "elapsed_s": round(time.time() - t0, 1),
        })

        # condition: combined (fine-tuned model + RAG, matching RAFT format)
        hits = retrieve(q["question"], embedder, index, chunks)
        top_score = hits[0]["score"] if hits else 0.0
        if top_score < RETRIEVAL_SCORE_THRESHOLD:
            answer, err = "[ABSTAINED: retrieval score below threshold]", None
        else:
            doc_block = format_doc_block(hits)
            prompt = build_prompt(q["question"], doc_block)
            t0 = time.time()
            answer, err = generate(tok, model, prompt)
        results.append({
            "id": q["id"], "category": q["category"], "condition": "combined",
            "question": q["question"],
            "answer": answer if err is None else f"[GENERATION ERROR: {err}]",
            "retrieval_hits": hits, "top_score": round(top_score, 3),
            "elapsed_s": round(time.time() - t0, 1),
        })

    release_model(tok, model)
    return results

In [ ]:
#MAIN --------------------------------------------------------

if __name__ == "__main__":
    embedder, index, chunks = load_retriever()

    base_results = run_base_conditions(embedder, index, chunks)
    with open(OUT_DIR / "base_and_rag_only.json", "w") as f:
        json.dump(base_results, f, indent=2)

    ft_results = run_finetuned_conditions(embedder, index, chunks)
    with open(OUT_DIR / "lora_only_and_combined.json", "w") as f:
        json.dump(ft_results, f, indent=2)

    all_results = base_results + ft_results
    with open(OUT_DIR / "all_results.json", "w") as f:
        json.dump(all_results, f, indent=2)

    print(f"Done. {len(all_results)} total generations across "
          f"{len(QUESTIONS)} questions x 4 conditions.")
    print(f"Saved to {OUT_DIR}")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Done. 68 total generations across 17 questions x 4 conditions.
Saved to /kaggle/working/ablation_results


In [ ]:
#Scoring Sheet Generation ----------------------------------------
"""
Turn all_results.json into a scoring sheet (CSV) for manual rating, plus
compute two automatable proxy metrics that don't require a human pass:

1. abstention_detected: keyword match for whether the model actually
   declined vs. answered. Useful to auto-flag your abstention probes
   (A1-A3) and the eponym-mismatch case (E1) for quick sanity-checking
   before you do the full manual read.
2. grounding_overlap: crude token-overlap between the generated answer
   and the retrieved chunk text, as a cheap (not a substitute for human
   judgment) proxy for "is this answer actually using the retrieved
   content or ignoring it." Only computed for rag_only / combined rows.

Manual columns to fill in per your plan (domain fluency, factual
grounding, hallucination rate) are left blank for you to score by hand
against the source papers -- that judgment call needs a human who knows
the chemistry, which is the right call here.
"""

import csv
import json
import re
from pathlib import Path

IN_PATH = Path("/kaggle/working/ablation_results/all_results.json")
OUT_PATH = Path("/kaggle/working/ablation_results/scoring_sheet.csv")

ABSTENTION_PHRASES = [
    "don't know", "do not know", "cannot determine", "can't determine",
    "not contain", "does not contain", "no information", "not mentioned",
    "not present in", "unable to answer", "insufficient context",
    "not addressed in", "not covered in",
]


def detect_abstention(answer: str) -> bool:
    a = answer.lower()
    if a.startswith("[abstained"):
        return True
    return any(p in a for p in ABSTENTION_PHRASES)


def token_overlap(answer: str, hits) -> float:
    if not hits:
        return 0.0
    ans_tokens = set(re.findall(r"[a-z0-9]+", answer.lower()))
    if not ans_tokens:
        return 0.0
    ctx_tokens = set()
    for h in hits:
        ctx_tokens |= set(re.findall(r"[a-z0-9]+", h["text"].lower()))
    if not ctx_tokens:
        return 0.0
    return round(len(ans_tokens & ctx_tokens) / len(ans_tokens), 3)


def main():
    with open(IN_PATH) as f:
        results = json.load(f)

    rows = []
    for r in results:
        abstained = detect_abstention(r["answer"])
        overlap = None
        if r["condition"] in ("rag_only", "combined") and r.get("retrieval_hits"):
            overlap = token_overlap(r["answer"], r["retrieval_hits"])

        rows.append({
            "id": r["id"],
            "category": r["category"],
            "condition": r["condition"],
            "question": r["question"],
            "answer": r["answer"],
            "top_retrieval_score": r.get("top_score", ""),
            "auto_abstention_detected": abstained,
            "auto_grounding_token_overlap": overlap if overlap is not None else "",
            # --- fill these in by hand against the source papers ---
            "domain_fluency_1to5": "",
            "factual_grounding_1to5": "",
            "hallucination_yes_no_partial": "",
            "notes": "",
        })

    fieldnames = list(rows[0].keys())
    with open(OUT_PATH, "w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)

    print(f"Wrote {len(rows)} rows to {OUT_PATH}")

    # Quick console summary of the automatable checks, grouped by condition,
    # so you can eyeball whether abstention behavior lines up with
    # expectation before doing the manual scoring pass.
    by_condition = {}
    for r, row in zip(results, rows):
        by_condition.setdefault(r["condition"], []).append(row["auto_abstention_detected"])

    print("\nAuto-abstention rate by condition:")
    for cond, flags in by_condition.items():
        rate = sum(flags) / len(flags)
        print(f"  {cond:10s}: {rate:.0%} ({sum(flags)}/{len(flags)})")


if __name__ == "__main__":
    main()

Wrote 68 rows to /kaggle/working/ablation_results/scoring_sheet.csv

Auto-abstention rate by condition:
  base      : 0% (0/17)
  rag_only  : 18% (3/17)
  lora_only : 6% (1/17)
  combined  : 53% (9/17)


In [ ]:
from huggingface_hub import login
login(token="HF_TOKEN")  # replace with your actual token or use Kaggle secrets

In [ ]:
"""
USAGE:
    !pip install gradio faiss-cpu pdfplumber -q
    # then run this script, or paste into a notebook cell
"""

import gc
import os
import re
import pickle

os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")

import gradio as gr
import numpy as np
import torch
import faiss
import pdfplumber
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer

# %% CONFIG -- identical to ablation_study.py's validated settings ------

FT_MODEL_ID = "aryxnsinhx/mistral-7b-microfluidics-raft"
DATASET_DIR = "/kaggle/input/datasets/aryxnsinhx1/ablation-aryxn"
INDEX_PATH = f"{DATASET_DIR}/microfluidics.index"
METADATA_PATH = f"{DATASET_DIR}/chunk_metadata.pkl"

RETRIEVAL_SCORE_THRESHOLD = 0.4
TOP_K = 5
MAX_CHUNK_CHARS_IN_PROMPT = 700

GEN_KWARGS = dict(
    do_sample=False,
    repetition_penalty=1.1,
    max_new_tokens=350,
)

SESSION_CHUNK_SIZE_CHARS = 900
SESSION_CHUNK_OVERLAP_CHARS = 150
MAX_SESSION_PAPERS = 8
MAX_SESSION_CHUNKS = 400  

EMBED_DIM = 768  

#runs once at startup ----------------------------------------

print("Loading embedder + core FAISS index...")
embedder = SentenceTransformer("all-mpnet-base-v2", device="cpu")
core_index = faiss.read_index(INDEX_PATH)
with open(METADATA_PATH, "rb") as f:
    raw = pickle.load(f)
core_chunks = raw.to_dict(orient="records") if hasattr(raw, "to_dict") else list(raw)
print(f"Loaded {core_index.ntotal} core vectors, {len(core_chunks)} chunk records.")

print("Loading fine-tuned model (fp16, no quantization)...")
tokenizer = AutoTokenizer.from_pretrained(FT_MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    FT_MODEL_ID,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True,
    attn_implementation="sdpa",
)
model.eval()
print("Model loaded.")


#new: PDF -> chunks -> session FAISS index --------------

def extract_pdf_text(filepath):
    """Extract text page by page, keeping page boundaries for section labels."""
    pages = []
    with pdfplumber.open(filepath) as pdf:
        for page in pdf.pages:
            text = page.extract_text() or ""
            pages.append(text)
    return pages


def guess_section_label(paragraph, page_num):
    """Light heuristic: use a header-like line if present, else fall back
    to a page reference. This is a starting point -- if your core corpus
    used a more rigorous section-detection step, swap it in here so
    session chunks carry the same style of section metadata."""
    first_line = paragraph.strip().split("\n", 1)[0].strip()
    looks_like_header = (
        0 < len(first_line) < 80
        and (first_line.isupper() or re.match(r"^\d+(\.\d+)*\.?\s+[A-Z]", first_line))
    )
    if looks_like_header:
        return first_line
    return f"page {page_num}"


def chunk_pdf_pages(pages, chunk_size=SESSION_CHUNK_SIZE_CHARS, overlap=SESSION_CHUNK_OVERLAP_CHARS):
    """Paragraph-aware chunking with overlap, mirroring the guidance we
    discussed: split on natural paragraph boundaries first, then pack into
    overlapping windows so information near a boundary isn't orphaned."""
    chunks = []
    for page_num, page_text in enumerate(pages, start=1):
        if not page_text.strip():
            continue
        paragraphs = [p.strip() for p in re.split(r"\n\s*\n", page_text) if p.strip()]
        buf = ""
        for para in paragraphs:
            candidate = (buf + "\n\n" + para).strip() if buf else para
            if len(candidate) <= chunk_size:
                buf = candidate
                continue
            if buf:
                chunks.append((buf, guess_section_label(buf, page_num)))
                buf = buf[-overlap:] + "\n\n" + para if overlap else para
            else:
                # single paragraph longer than chunk_size: hard-split it
                for i in range(0, len(para), chunk_size - overlap):
                    piece = para[i:i + chunk_size]
                    chunks.append((piece, guess_section_label(piece, page_num)))
                buf = ""
        if buf:
            chunks.append((buf, guess_section_label(buf, page_num)))
    return chunks


def ingest_pdf(filepath, paper_title, session_state):
    session_state = session_state or {"index": None, "chunks": [], "papers": []}

    if len(session_state["papers"]) >= MAX_SESSION_PAPERS:
        return session_state, (
            f"Session cap reached ({MAX_SESSION_PAPERS} papers). "
            "Reset the session to ingest more."
        )

    try:
        pages = extract_pdf_text(filepath)
    except Exception as e:
        return session_state, f"Failed to read PDF: {e}"

    raw_chunks = chunk_pdf_pages(pages)
    if not raw_chunks:
        return session_state, "No extractable text found in this PDF (is it scanned/image-only?)."

    remaining_budget = MAX_SESSION_CHUNKS - len(session_state["chunks"])
    if remaining_budget <= 0:
        return session_state, f"Session chunk cap reached ({MAX_SESSION_CHUNKS}). Reset to continue."
    raw_chunks = raw_chunks[:remaining_budget]

    texts = [c[0] for c in raw_chunks]
    embs = embedder.encode(texts, normalize_embeddings=True, show_progress_bar=False)
    embs = np.array(embs, dtype="float32")

    if session_state["index"] is None:
        session_state["index"] = faiss.IndexFlatIP(EMBED_DIM)

    session_state["index"].add(embs)
    for text, section in raw_chunks:
        session_state["chunks"].append({
            "paper_title": paper_title,
            "section": section,
            "text": text,
        })
    session_state["papers"].append({"title": paper_title, "n_chunks": len(raw_chunks)})

    status = (
        f"Ingested \"{paper_title}\": {len(raw_chunks)} chunks added. "
        f"Session total: {len(session_state['chunks'])} chunks across "
        f"{len(session_state['papers'])} paper(s)."
    )
    return session_state, status


def corpus_status_md(session_state):
    if not session_state or not session_state.get("papers"):
        return "_No papers uploaded this session. Answers are grounded in the core corpus only._"
    lines = ["**Session-uploaded papers (this browser session only):**\n"]
    for p in session_state["papers"]:
        lines.append(f"- {p['title']} — {p['n_chunks']} chunks")
    return "\n".join(lines)


def reset_session(session_state):
    return {"index": None, "chunks": [], "papers": []}, "_Session cleared. Core corpus only._", ""


# %% RETRIEVAL / GENERATION -- core logic unchanged, now source-merged --

def retrieve(query, session_state, top_k=TOP_K):
    q_emb = np.array(embedder.encode([query], normalize_embeddings=True), dtype="float32")

    hits = []

    scores, idxs = core_index.search(q_emb, top_k)
    for score, idx in zip(scores[0], idxs[0]):
        if idx == -1:
            continue
        c = core_chunks[idx]
        hits.append({
            "score": float(score),
            "paper_title": c["paper_title"],
            "section": c["section"],
            "text": c["text"],
            "source": "core corpus",
        })

    if session_state and session_state.get("index") is not None and session_state["index"].ntotal > 0:
        s_scores, s_idxs = session_state["index"].search(q_emb, top_k)
        for score, idx in zip(s_scores[0], s_idxs[0]):
            if idx == -1:
                continue
            c = session_state["chunks"][idx]
            hits.append({
                "score": float(score),
                "paper_title": c["paper_title"],
                "section": c["section"],
                "text": c["text"],
                "source": f"uploaded: {c['paper_title']}",
            })

    hits.sort(key=lambda h: h["score"], reverse=True)
    return hits[:top_k]


def format_doc_block(hits):
    blocks = []
    for i, h in enumerate(hits, start=1):
        text = h["text"]
        if len(text) > MAX_CHUNK_CHARS_IN_PROMPT:
            text = text[:MAX_CHUNK_CHARS_IN_PROMPT].rsplit(" ", 1)[0] + "..."
        blocks.append(f"[DOC {i}] ({h['paper_title']}, {h['section']}) {text}")
    return "\n\n".join(blocks)


def build_prompt(question, doc_block):
    return (
        "You are a research assistant. Use only the context below to answer. "
        "If the context does not contain the answer, say you don't know.\n\n"
        f"Context:\n{doc_block}\n\nQuestion: {question}\n[/INST]"
    )


def truncate_at_first_repeat(text):
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    seen, kept = set(), []
    for s in sentences:
        k = s.strip().lower()
        if k in seen and len(k) > 15:
            break
        seen.add(k)
        kept.append(s)
    return " ".join(kept)


def generate(prompt):
    inputs = None
    out = None
    try:
        inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
        with torch.inference_mode():
            out = model.generate(**inputs, **GEN_KWARGS, pad_token_id=tokenizer.eos_token_id)
        text = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
        return truncate_at_first_repeat(text), None
    except torch.cuda.OutOfMemoryError as e:
        return None, f"OOM: {e}"
    finally:
        del inputs, out
        gc.collect()
        torch.cuda.empty_cache()


#DEMO LOGIC -----------------------------------------------------------

SOURCE_BADGE_CSS = {
    "core": "background:#2D3A4A; color:#9FD3FF;",
    "upload": "background:#3A2D4A; color:#D9B8FF;",
}


def badge_for(source):
    style = SOURCE_BADGE_CSS["upload"] if source.startswith("uploaded:") else SOURCE_BADGE_CSS["core"]
    return f'<span style="{style} padding:2px 8px; border-radius:10px; font-size:11px;">{source}</span>'


def answer_question(question, session_state):
    """Core retrieve-then-generate call. Returns (answer_text, citation_markdown)."""
    if not question or not question.strip():
        return "", ""

    hits = retrieve(question, session_state)
    top_score = hits[0]["score"] if hits else 0.0

    if top_score < RETRIEVAL_SCORE_THRESHOLD:
        answer = (
            f"I don't have a confident enough match in the corpus to answer this "
            f"reliably. (Top retrieval score: {top_score:.2f}, below the "
            f"{RETRIEVAL_SCORE_THRESHOLD:.2f} threshold.)"
        )
        citation_md = "_No sources met the confidence threshold for this query._"
        return answer, citation_md

    doc_block = format_doc_block(hits)
    prompt = build_prompt(question, doc_block)
    answer, err = generate(prompt)
    if err:
        answer = f"Generation failed: {err}"

    citation_lines = ["**Sources retrieved:**"]
    for i, h in enumerate(hits, start=1):
        citation_lines.append(
            f"{i}. {badge_for(h['source'])} *{h['paper_title']}* — {h['section']} "
            f"(similarity: {h['score']:.2f})"
        )
    citation_md = "\n".join(citation_lines)

    return answer, citation_md


EXAMPLES = [
    "What physical mechanism drives mixing in a microfluidic channel at low Reynolds number?",
    "What causes electrohydrodynamic instability at a liquid-liquid interface under an applied electric field?",
    "How is a diazonium salt used to detect bilirubin in a sample, chemically speaking?",
    "Explain the Van den Bergh reaction and how it distinguishes conjugated from unconjugated bilirubin.",
]

def latest_sources_md(session_state, last_hits):
    """Right-panel content: sources for the most recent answer only,
    rendered as cards rather than buried in the chat transcript."""
    if not last_hits:
        return "_Ask a question to see grounding sources here._"
    lines = []
    for i, h in enumerate(last_hits, start=1):
        lines.append(
            f"**{i}.** {badge_for(h['source'])}<br>"
            f"*{h['paper_title']}* — {h['section']}<br>"
            f"similarity: `{h['score']:.2f}`\n"
        )
    return "\n\n".join(lines)


def chat_respond_v2(question, chat_history, session_state):
    """Same as chat_respond but also returns the raw hit list so the
    right-hand sources panel can render independently of the chat bubble."""
    chat_history = chat_history or []
    if not question or not question.strip():
        return chat_history, "", session_state, latest_sources_md(session_state, None)

    hits = retrieve(question, session_state)
    top_score = hits[0]["score"] if hits else 0.0

    if top_score < RETRIEVAL_SCORE_THRESHOLD:
        answer = (
            f"I don't have a confident enough match in the corpus to answer this "
            f"reliably. (Top retrieval score: {top_score:.2f}, below the "
            f"{RETRIEVAL_SCORE_THRESHOLD:.2f} threshold.)"
        )
        hits_for_panel = hits
    else:
        doc_block = format_doc_block(hits)
        prompt = build_prompt(question, doc_block)
        answer, err = generate(prompt)
        if err:
            answer = f"Generation failed: {err}"
        hits_for_panel = hits

    chat_history = chat_history + [
        {"role": "user", "content": question},
        {"role": "assistant", "content": answer},
    ]
    return chat_history, "", session_state, latest_sources_md(session_state, hits_for_panel)


CUSTOM_CSS = """
html, body { height: 100vh !important; overflow: hidden !important; }
.gradio-container { max-width: 1280px !important; height: 100vh !important; margin: auto !important; padding-top: 8px !important; overflow: hidden !important; }
#header-bar { display: flex !important; align-items: center !important; justify-content: space-between !important; margin-bottom: 0 !important; }
#domain-line p { margin: 2px 0 !important; font-size: 13px !important; line-height: 1.4 !important; }
#domain-line h2 { margin-bottom: 2px !important; font-size: 28px !important; }
#corpus-pill { font-size: 12px !important; padding: 3px 10px !important; border-radius: 12px !important; background: rgba(120,120,120,0.12) !important; }
.accordion { margin: 4px 0 !important; }
#chatbot { height: 46vh !important; }
#chatbot .message { font-family: Georgia, serif !important; font-size: 14px !important; line-height: 1.45 !important; }
#sources-panel { border-left: 1px solid rgba(120,120,120,0.25); padding-left: 18px !important; font-size: 12px !important; max-height: 46vh !important; overflow-y: auto !important; }
#sources-title { font-weight: 600 !important; margin-bottom: 6px !important; }
footer { display: none !important; }
"""

with gr.Blocks(css=CUSTOM_CSS, title="Microfluidics RAFT Research Assistant", theme=gr.themes.Soft()) as demo:
    session_state = gr.State({"index": None, "chunks": [], "papers": []})

    # ---- Top bar: title + live corpus status + collapsed upload -------
    with gr.Row(elem_id="header-bar"):
        gr.Markdown(
            "## Microfluidics & Bilirubin Detection Research Assistant\n"
            "RAFT-tuned Mistral-7B + retrieval, with live paper uploads layered in per session.\n\n"
            "**Domain expertise:** microfluidic mixing & electrohydrodynamic instability · "
            "Micro-channel device fabrication · microfluidic reaction kinetics · Complex microfluidic system · "
            "related fluid dynamics & electrochemistry literature",
            elem_id="domain-line",
        )
    corpus_status = gr.Markdown(corpus_status_md(None), elem_id="corpus-pill")

    with gr.Accordion("Manage session corpus (upload a paper / reset)", open=False):
        with gr.Row():
            with gr.Column(scale=2):
                pdf_upload = gr.File(label="PDF file", file_types=[".pdf"])
                paper_title_box = gr.Textbox(
                    label="Paper title (used in citations)",
                    placeholder="e.g. Smith et al. 2023, Microchannel Mixing",
                )
                with gr.Row():
                    ingest_btn = gr.Button("Ingest into session corpus", variant="primary", size="sm")
                    reset_btn = gr.Button("Reset session", variant="stop", size="sm")
                upload_status = gr.Markdown(elem_id="upload-status")
            with gr.Column(scale=1):
                gr.Markdown(
                    "Uploaded PDFs are chunked/embedded the same way as the core "
                    "903-chunk corpus and added to a **session-only** index — the "
                    "validated core index is never modified. Sources are labeled "
                    "distinctly in the panel to the right of the chat."
                )

    # ---- Main row: chat (wide) + live sources panel (narrow) ----------
    with gr.Row():
        with gr.Column(scale=3):
            chatbot = gr.Chatbot(
                label=None, elem_id="chatbot", type="messages",
                show_label=False, avatar_images=(None, None),
            )
            with gr.Row():
                question_box = gr.Textbox(
                    label=None, show_label=False,
                    placeholder="Ask a question about the corpus...",
                    scale=5, container=False,
                )
                submit_btn = gr.Button("Send", variant="primary", scale=1)

        with gr.Column(scale=1, min_width=260, elem_id="sources-panel"):
            gr.Markdown("**Sources for last answer**", elem_id="sources-title")
            sources_panel = gr.Markdown(latest_sources_md(None, None))

    # --- wiring ---
    submit_btn.click(
        chat_respond_v2,
        inputs=[question_box, chatbot, session_state],
        outputs=[chatbot, question_box, session_state, sources_panel],
    )
    question_box.submit(
        chat_respond_v2,
        inputs=[question_box, chatbot, session_state],
        outputs=[chatbot, question_box, session_state, sources_panel],
    )

    def _ingest_and_report(file, title, state):
        if file is None:
            return state, "Please choose a PDF first.", corpus_status_md(state)
        title = title.strip() or os.path.basename(file.name)
        new_state, status = ingest_pdf(file.name, title, state)
        return new_state, status, corpus_status_md(new_state)

    ingest_btn.click(
        _ingest_and_report,
        inputs=[pdf_upload, paper_title_box, session_state],
        outputs=[session_state, upload_status, corpus_status],
    )

    reset_btn.click(reset_session, inputs=[session_state], outputs=[session_state, corpus_status, upload_status])

if __name__ == "__main__":
    demo.launch(share=True)

Loading embedder + core FAISS index...


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

Model loaded.


/tmp/ipykernel_58/3237218753.py:421: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Microfluidics RAFT Research Assistant", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_58/3237218753.py:421: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(css=CUSTOM_CSS, title="Microfluidics RAFT Research Assistant", theme=gr.themes.Soft()) as demo:
/tmp/ipykernel_58/3237218753.py:459: DeprecationWarning: The default value of 'allow_tags' in gr.Chatbot will be changed from False to True in Gradio 6.0. You will need to explicitly set allow_tags=False if you want to disable tags in your chatbot.
  chatbot = gr.Chatbot(


* Running on local URL:  http://127.0.0.1:7860
* Running on public URL: https://8c5667d25558b76803.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
